# 04 · Synthesis — Indonesia GROW deep-dive

Pulls `01`–`03` (what/when Indonesia grows) and `05` (the cassava→wheat substitution lever) into
one findings page: **what the GROW production data establishes, and what it rules out.**

The framing the deck argues — *problem → mechanism → solution* — maps onto this track as:
the **problem** is a structural production zero (wheat), the **mechanism** is that Indonesia's
domestic staple base is shrinking on *area* while intensifying on *yield*, and the **solution**
(cassava → MOCAF) is therefore bounded by land retention, not agronomy.

## Setup

In [1]:
import sys; sys.path.append('..')
import pandas as pd
from src.load import load_indonesia, load_qcl_world
from src.clean import drop_item_aggregates

qcl = load_indonesia('QCL')
LATEST = qcl['year'].max()
BASE   = qcl['year'].min()
CASSAVA = 'Cassava, fresh'

[cache] QCL_indonesia_2010_2024.parquet


## Headline numbers

In [2]:
prod = drop_item_aggregates(qcl[(qcl['Element']=='Production') & (qcl['year']==LATEST)])
top5 = prod.groupby('Item')['value'].sum().sort_values(ascending=False).head(5)
print(f'Top 5 crops by production, {LATEST}:')
print(top5)

Top 5 crops by production, 2024:
Item
Oil palm fruit              2.430000e+08
Hen eggs in shell, fresh    1.527872e+08
Rice                        5.314273e+07
Sugar cane                  3.200000e+07
Coconuts, in shell          1.798539e+07
Name: value, dtype: float64


In [3]:
# Area-vs-yield decomposition for the crops the story rests on — the single pattern
# that runs through 02, 03 and 05. (Same method as nb 02's rice cell.)
def decompose(item):
    p = (qcl[(qcl['Item'] == item) &
             (qcl['Element'].isin(['Production', 'Area harvested', 'Yield']))]
         .pivot_table(index='year', columns='Element', values='value').sort_index())
    return {'Item': item,
            **{f'{c} %chg': 100 * (p[c].loc[LATEST] / p[c].loc[BASE] - 1)
               for c in ['Production', 'Area harvested', 'Yield']}}

recap = pd.DataFrame([decompose(i) for i in
                      ['Rice', 'Maize (corn)', CASSAVA, 'Soya beans']]).set_index('Item')
print(f'{BASE} -> {LATEST} change, %:')
print(recap.round(1).to_string())
print('\nEvery domestic staple: production down or flat, AREA down hard, YIELD up.')
print('Indonesia is not farming worse — it is farming less land.')

2010 -> 2024 change, %:
                Production %chg  Area harvested %chg  Yield %chg
Item                                                            
Rice                      -10.4                -14.8         5.3
Maize (corn)              -17.4                -38.3        33.9
Cassava, fresh            -34.7                -53.2        39.6
Soya beans                -74.9                -79.8        24.2

Every domestic staple: production down or flat, AREA down hard, YIELD up.
Indonesia is not farming worse — it is farming less land.


In [4]:
# The bound on the cassava solution (detail in nb 05 §5): Indonesia is already near the
# achievable cassava-yield frontier, so the headroom is finite and the area loss is bigger.
cas_world = load_qcl_world(CASSAVA, ['Yield', 'Area harvested'])
wpiv = (cas_world[cas_world['year'] == LATEST]
        .pivot_table(index='Area', columns='Element', values='value').dropna())
peers = wpiv[wpiv['Area harvested'] >= 50_000].sort_values('Yield', ascending=False)

idn_y = peers.loc['Indonesia', 'Yield']
front, front_y = peers.index[0], peers.iloc[0]['Yield']
cur_area = peers.loc['Indonesia', 'Area harvested']
area_lost = (qcl[(qcl['Item'] == CASSAVA) & (qcl['Element'] == 'Area harvested')]
             .set_index('year')['value'].loc[BASE]) - cur_area

print(f'Cassava yield rank: Indonesia #{list(peers.index).index("Indonesia")+1} of {len(peers)} '
      f'peer producers (\u2265 50k ha)')
print(f'  Indonesia {idn_y:,.0f} kg/ha vs frontier {front} {front_y:,.0f} kg/ha '
      f'\u2014 gap {100*(front_y-idn_y)/front_y:.1f}%')
print(f'  Lever A, close that gap on today\'s area: +{(front_y-idn_y)*cur_area/1e9:,.2f} Mt')
print(f'  Lever B, recover the {area_lost:,.0f} ha lost since {BASE}: '
      f'+{area_lost*idn_y/1e9:,.2f} Mt  <- the bigger lever')

[cache] QCL_world_cassava_fresh_2010_2024.parquet
Cassava yield rank: Indonesia #2 of 40 peer producers (≥ 50k ha)
  Indonesia 28,230 kg/ha vs frontier India 35,574 kg/ha — gap 20.6%
  Lever A, close that gap on today's area: +4.06 Mt
  Lever B, recover the 629,597 ha lost since 2010: +17.77 Mt  <- the bigger lever


## Findings

### 1. What Indonesia grows — and the one thing it doesn't
Oil palm dominates tonnage (243 Mt) and value ($32.3bn), but the **calorie** story is rice
(53.1 Mt, $21.3bn) with maize (15.1 Mt) and cassava (15.6 Mt) behind it. Against that:
**Indonesia has zero wheat records in QCL, in every year 2010–2024** — not a small number,
literally absent. That is the production-side root of being the world's #1 wheat importer:
there is no domestic wheat sector to scale, only a substitution question.

### 2. When / how intensively — the area story is the real story
- **Cropping intensity fell from 114% (2015) to 96% (2024)** (area harvested ÷ arable land, from RL).
  Above 100% means multi-season cropping; dropping below it means Indonesia stopped double-cropping
  land it used to double-crop. This is a proxy, not a calendar — FAOSTAT has no planting/harvest
  months (see CLAUDE.md).
- **Every domestic staple lost area 2015→2024**: rice −1.34M ha (−11.8%), maize −1.24M ha (−32.7%),
  soya beans −0.48M ha (−78.3%), cassava −0.40M ha (−41.7%). What grew instead was
  non-staple/export-facing: nutmeg, chillies, sugar cane, shallots, tobacco, avocados.
- **Yield rose everywhere at the same time** (rice +5.3%, cassava +39.6% since 2010). So the decline
  is not agronomic failure — it is land leaving staple production, partly masked by intensification.

### 3. The cassava → MOCAF lever: motivated by GROW, and bounded by it (nb 05)
- **Base is shrinking**: cassava production −34.7% (23.9 → 15.6 Mt, 2010→2024), driven by a
  **−53.2% area** collapse (1.18M → 0.55M ha) against a **+39.6% yield** gain.
  `corr(area, production) = +0.97`. The proposed solution's raw material is itself under pressure.
- **Yield headroom is real but small**: Indonesia is already the **#2 cassava yielder of 40 peer
  producers** (≥50k ha), 28.2 t/ha vs India's 35.6 — a **20.6% gap**, and *at parity with the
  peer top-5 mean*. Closing it entirely on today's area adds **+4.06 Mt, no new land**.
- **Area is the 4.4× bigger lever**: the 630k ha cassava already lost since 2010, at today's yield,
  is **+17.8 Mt** — more than the entire current crop.
- **Offset ceiling**: all 15.6 Mt of cassava converted to MOCAF is 3.12–5.41 Mt flour = **32–55% of
  food-wheat demand** (9.8 MMT). A ceiling, not a forecast — cassava already has food/feed/starch uses.
- **Achievable band**: a **10% blend** needs 2.83–4.90 Mt fresh cassava (18–31% of the crop) and avoids
  0.98 Mt of wheat (~$0.29bn/yr at $300/t). A **20% blend** needs 5.66–9.80 Mt (36–63% of the crop) —
  not reachable without either new land or a large diversion from existing uses.
- **The one number that carries the recommendation**: closing the yield gap alone funds an
  **8.3–14.4% blend from new cassava only** — no new land, no diversion. Processing yield is what
  decides which end of that range Indonesia gets.

### Data gaps to close if this becomes the project focus
- **Month-level crop calendar** (FAO Crop Calendar tool) for a true "when" answer — QCL/QI/QV/RL are annual.
- **Provincial/island breakdown** — FAOSTAT is national-only, and Indonesia's agriculture varies hugely
  by island (Java rice, multi-season; Sumatra/Kalimantan oil palm). Any island claim must come from
  outside this data.
- **Why cassava area fell** — competing land use (oil palm? urbanisation?) is a SUSTAIN/land-cover
  question, not answerable in QCL. It is the single biggest determinant of whether MOCAF can scale.
- **Import prices** — the $-saved column in nb 05's slide-8 matrix rests on a $300/t CIF assumption;
  the TRADE track should replace it with observed unit values.

## Recommendation

> **Go — with the scope narrowed.** GROW data supports the wheat-vulnerability thesis and, more usefully,
> **bounds the proposed fix**. Cassava is the credible domestic lever: Indonesia is the world's #6 cassava
> producer and already near the achievable yield frontier, so this is not a "teach them to farm" story.
>
> The binding constraint is **area decline plus processing yield, not agronomy.** Cassava lost 53% of its
> area in 14 years; yield gains have been running hard just to slow the fall. Closing the remaining yield
> gap is worth +4.06 Mt and funds an 8–14% wheat-flour blend from new cassava alone — the strongest,
> land-neutral claim in the deck — but it cannot fund the 20% blend, and it is 4.4× smaller than simply
> not losing more cassava land.
>
> **So the policy ask is two-sided and should be presented that way:** (1) **land retention** for cassava,
> which is where the tonnage actually is, and (2) **processing-yield upgrade** from ~20% community
> sun-dried toward ~34.6% lab-grade MOCAF, which nearly halves the cassava needed per unit of wheat
> displaced. Both line up with the MOCAF processing and farmer-regeneration barriers the SUSTAIN research
> already identified — GROW's contribution is showing they are the *binding* constraints, and pricing them.

In [5]:
print('Synthesis complete.')

Synthesis complete.
